In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv('../data/raw/tokyo.csv')

In [3]:
df["jnc"] = df["Links to Listing"].str.extract(r"jnc_(\d+)")

In [4]:
# Duplicate values =8002 (expected)

phys = ["Floor", "Rooms", "Square Meters"]
d = df.drop_duplicates()                          # exact copies: expect 8,002 removed
n_exact = len(df) - len(d)

bad = d.groupby("jnc")[phys].nunique().max(axis=1)
conflict = bad[bad > 1].index                     # jnc values with physical conflicts
d = d[~d["jnc"].isin(conflict)]
n_conflict = len(df) - n_exact - len(d)

d = d.drop_duplicates(subset="jnc", keep="first")
print(n_exact, n_conflict, len(d))


8002 10 183490


In [5]:
d.to_csv("../data/interim/listings_dedup.csv", index=False, encoding="utf-8-sig")
print(len(d))

183490


In [6]:
df=pd.read_csv('../data/interim/listings_dedup.csv')

In [ ]:
def parse_yen(text):
    if text == "-":
        return None
    if text.endswith("万円"):
        number = text.replace("万円", "")      # "6.5万円" -> "6.5"
        return round(float(number) * 10000)
    if text.endswith("円"):
        number = text.replace("円", "")        # "3000円" -> "3000"
        return int(number)
    return "FAILED"                            # anything unexpected                    # anything unexpected

In [8]:
print(parse_yen("6.5万円"))    # expect 65000
print(parse_yen("10.2万円"))   # expect 102000
print(parse_yen("3000円"))     # expect 3000
print(parse_yen("-"))          # expect None

65000
102000
3000
None


In [9]:
d["Price_yen"] = d["Price"].apply(parse_yen)
print((d["Price_yen"] == "FAILED").sum())

0


In [10]:
d["Administrative Costs_yen"] = d["Administrative Costs"].apply(parse_yen)
print((d["Administrative Costs_yen"] == "FAILED").sum())

0


In [11]:
d["Deposit_yen"] = d["Deposit"].apply(parse_yen)
print((d["Deposit_yen"] == "FAILED").sum())

0


In [12]:
d["Gratuity_yen"] = d["Gratuity"].apply(parse_yen)
print((d["Gratuity_yen"] == "FAILED").sum())

0


In [13]:
d["Admin_yen"] = d["Administrative Costs_yen"]  # optional rename
d["Administrative Costs_yen"].describe()

count    150179.000000
mean       6101.231497
std        4356.322585
min           1.000000
25%        3000.000000
50%        5000.000000
75%       10000.000000
max      295000.000000
Name: Administrative Costs_yen, dtype: float64

In [14]:
cols = ["Price_yen", "Administrative Costs_yen", "Deposit_yen", "Gratuity_yen"]
for c in cols:
    d[c] = pd.to_numeric(d[c], errors="coerce")

print(d[cols].describe())
print(d[cols].isna().sum())

          Price_yen  Administrative Costs_yen   Deposit_yen  Gratuity_yen
count  1.834900e+05             150179.000000  1.297200e+05  1.129970e+05
mean   9.931687e+04               6101.231497  1.275848e+05  1.164612e+05
std    7.756559e+04               4356.322585  2.375469e+05  9.100306e+04
min    1.060000e+04                  1.000000  1.000000e+02  1.000000e+02
25%    6.300000e+04               3000.000000  6.600000e+04  7.100000e+04
50%    8.250000e+04               5000.000000  8.750000e+04  9.200000e+04
75%    1.130000e+05              10000.000000  1.280000e+05  1.300000e+05
max    4.450000e+06             295000.000000  1.780000e+07  3.300000e+06
Price_yen                       0
Administrative Costs_yen    33311
Deposit_yen                 53770
Gratuity_yen                70493
dtype: int64


In [15]:
pd.options.display.float_format = "{:,.0f}".format
print(d[["Price_yen", "Administrative Costs_yen", "Deposit_yen", "Gratuity_yen"]].describe())

       Price_yen  Administrative Costs_yen  Deposit_yen  Gratuity_yen
count    183,490                   150,179      129,720       112,997
mean      99,317                     6,101      127,585       116,461
std       77,566                     4,356      237,547        91,003
min       10,600                         1          100           100
25%       63,000                     3,000       66,000        71,000
50%       82,500                     5,000       87,500        92,000
75%      113,000                    10,000      128,000       130,000
max    4,450,000                   295,000   17,800,000     3,300,000


In [16]:
pd.options.display.float_format = "{:,.0f}".format
cols = ["Listing Title", "Price", "Administrative Costs", "Deposit", "Gratuity", "Square Meters", "Rooms"]

print(d.nsmallest(5, "Administrative Costs_yen")[cols])
print(d.nlargest(5, "Price_yen")[cols])
print(d.nlargest(5, "Deposit_yen")[cols])
print((d["Deposit_yen"] < 1000).sum(), (d["Gratuity_yen"] < 1000).sum())

              Listing Title  Price Administrative Costs Deposit Gratuity  \
32234   ＪＲ山手線 田町駅 10階建 築16年  8.3万円                   1円       -    8.3万円   
83203   西武新宿線 下井草駅 2階建 築41年   11万円                   1円    11万円     11万円   
94759                カインドピア  5.7万円                   1円   5.7万円    5.7万円   
52065            パークアクシス御成門   32万円                   2円    64万円     64万円   
142227                  山文荘  3.1万円                 100円   3.1万円    3.1万円   

       Square Meters  Rooms  
32234         21.28m     1K  
83203         43.69m     3K  
94759            20m  ワンルーム  
52065         55.63m   1LDK  
142227         12.4m  ワンルーム  
       Listing Title  Price Administrative Costs Deposit Gratuity  \
110731   六本木ヒルズレジデンス  445万円                    -  1780万円        -   
178787    ＨＩＲＯＯ　９－２０  345万円                    -  1380万円        -   
160226         二番町戸建  330万円                    -   660万円    330万円   
117342      ラ・トゥール渋谷  320万円                    -   960万円        -   
30030         ガーデン

In [17]:
d["deposit_months"] = d["Deposit_yen"] / d["Price_yen"]
d["gratuity_months"] = d["Gratuity_yen"] / d["Price_yen"]
print(d[["deposit_months", "gratuity_months"]].describe())

       deposit_months  gratuity_months
count         129,720          112,997
mean                1                1
std                 0                0
min                 0                0
25%                 1                1
50%                 1                1
75%                 1                1
max                24               10


In [18]:
pd.reset_option("display.float_format")
print(d["deposit_months"].round(2).value_counts().head(10))
print(d["gratuity_months"].round(2).value_counts().head(10))

deposit_months
1.00    116675
2.00      9333
0.50      1779
3.00       465
1.50       404
4.00       130
0.75        43
0.74        32
1.25        24
6.00        23
Name: count, dtype: int64
gratuity_months
1.00    101785
2.00      5190
0.50      2102
1.50      1792
1.20       692
1.70       136
1.10        72
0.54        33
0.48        31
0.52        29
Name: count, dtype: int64


In [19]:
print((d["deposit_months"] > 3).sum(), (d["deposit_months"] < 0.1).sum())
print((d["gratuity_months"] > 3).sum(), (d["gratuity_months"] < 0.1).sum())

174 5
14 13


In [20]:
d.columns

Index(['Listing Title', 'Area', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months'],
      dtype='str')

In [21]:
d["Price_yen"].describe()

count    1.834900e+05
mean     9.931687e+04
std      7.756559e+04
min      1.060000e+04
25%      6.300000e+04
50%      8.250000e+04
75%      1.130000e+05
max      4.450000e+06
Name: Price_yen, dtype: float64

In [22]:
d.head()

,Listing Title,Area,Distances to Railway Stations,Building Age,Stories Tall,Floor,Price,Administrative Costs,Deposit,Gratuity,...,Square Meters,Links to Listing,jnc,Price_yen,Administrative Costs_yen,Deposit_yen,Gratuity_yen,Admin_yen,deposit_months,gratuity_months
0,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,47.32m,https://suumo.jp/chintai/jnc_000055105229/?bc=...,000055105229,65000,3000.0,NaN,NaN,3000.0,NaN,NaN
1,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,47.32m,https://suumo.jp/chintai/jnc_000055105230/?bc=...,000055105230,65000,3000.0,NaN,NaN,3000.0,NaN,NaN
2,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.8万円,3000円,-,-,...,51.06m,https://suumo.jp/chintai/jnc_000054648287/?bc=...,000054648287,68000,3000.0,NaN,NaN,3000.0,NaN,NaN
3,カーザ コリーナ,東京都青梅市野上町２,ＪＲ青梅線/河辺駅 歩15分,築5年,3階建,3階,8.5万円,4000円,8.5万円,4.25万円,...,57.29m,https://suumo.jp/chintai/jnc_000055414464/?bc=...,000055414464,85000,4000.0,85000.0,42500.0,4000.0,1.0,0.5
4,京王相模原線 京王よみうりランド駅 2階建 築14年,東京都稲城市矢野口,"京王相模原線/京王よみうりランド駅 歩10分,ＪＲ南武線/矢野口駅 歩17分,京王相模原線/...",築14年,2階建,1階,8.1万円,3900円,-,-,...,42.49m,https://suumo.jp/chintai/jnc_000037983769/?bc=...,000037983769,81000,3900.0,NaN,NaN,3900.0,NaN,NaN


In [23]:
cols = ["Listing Title", "Area", "Distances to Railway Stations", "Building Age",
        "Stories Tall", "Floor", "Price", "Administrative Costs",
        "Deposit", "Gratuity", "Rooms", "Square Meters"]
n = d.duplicated(subset=cols, keep=False).sum()
print(n)

36894


In [24]:
d["lookalike_id"] = d.groupby(cols).ngroup()
sizes = d.groupby("lookalike_id").size()
print(sizes.value_counts().sort_index().head(10))
print(sizes.max())

1     146596
2      11530
3       2131
4        775
5        293
6        164
7         80
8         27
9         36
10        17
Name: count, dtype: int64
30


In [25]:
d.to_csv("../data/interim/listings_parsed.csv", index=False, encoding="utf-8-sig")